This notebook tests all models (from huggingface) on all steps validation sets. this notebook was run in Google Colab using L4 GPU. This same notebook was used for both experiemntal groups. 

In [ ]:
!pip install -q bitsandbytes accelerate

In [ ]:
import json
import torch
import os
import pandas as pd
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModelForSeq2SeqLM
from huggingface_hub import login

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Login to Hugging Face
hf_token = 'hf_token'
login(token=hf_token)

# Dictionary of your saved models on Hugging Face
model_families = {
  "neo1.3": {
        "step_1": "ehoangsimon/gpt-neo-1_3b-step1", # this model is the same for both test groups
        "step_2": "ehoangsimon/gpt-neo-1_3b-replay-step2",
        "step_3": "ehoangsimon/gpt-neo-1_3b-replay-step3",
        "step_4": "ehoangsimon/gpt-neo-1_3b-replay-step4",
        "step_5": "ehoangsimon/gpt-neo-1_3b-replay-step5",
        "step_6": "ehoangsimon/gpt-neo-1_3b-replay-step6",
        "step_7": "ehoangsimon/gpt-neo-1_3b-replay-step7",
        "step_8": "ehoangsimon/gpt-neo-1_3b-replay-step8",
        "step_9": "ehoangsimon/gpt-neo-1_3b-replay-step9",
        "step_10":"ehoangsimon/gpt-neo-1_3b-replay-step10",
    },
    "pythia2.8": {
        "step_1": "ehoangsimon/pythia_2.8_step1",     # this model is the same for both test groups
        "step_2": "ehoangsimon/pythia-2_8b-replay-step2",
        "step_3": "ehoangsimon/pythia-2_8b-replay-step3",
        "step_4": "ehoangsimon/pythia-2_8b-replay-step4",
        "step_5": "ehoangsimon/pythia-2_8b-replay-step5",
        "step_6": "ehoangsimon/pythia-2_8b-replay-step6",
        "step_7": "ehoangsimon/pythia-2_8b-replay-step7",
        "step_8": "ehoangsimon/pythia-2_8b-replay-step8",
        "step_9": "ehoangsimon/pythia-2_8b-replay-step9",
        "step_10": "ehoangsimon/pythia-2_8b-replay-step10"
    },
    "t5_base": {
        1: "ehoangsimon/t5-merged-step1",   # this model is the same for both test groups
        2: "ehoangsimon/t5-replay-step2",
        3: "ehoangsimon/t5-replay-step3",
        4: "ehoangsimon/t5-replay-step4",
        5: "ehoangsimon/t5-replay-step5",
        6: "ehoangsimon/t5-replay-step6",
        7: "ehoangsimon/t5-replay-step7",
        8: "ehoangsimon/t5-replay-step8",
        9: "ehoangsimon/t5-replay-step9",
        10: "ehoangsimon/t5-replay-step10",
    },
    "phi_1.5": {
        1: "ehoangsimon/phi1_5-step1",              # this model is the same for both test groups
        2: "ehoangsimon/phi15-replay-step2",
        3: "ehoangsimon/phi15-replay-step3",
        4: "ehoangsimon/phi15-replay-step4",
        5: "ehoangsimon/phi15-replay-step5",
        6: "ehoangsimon/phi15-replay-step6",
        7: "ehoangsimon/phi15-replay-step7",
        8: "ehoangsimon/phi15-replay-step8",
        9: "ehoangsimon/phi15-replay-step9",
        10: "ehoangsimon/phi15-replay-step10",
    }
}


DATA_ROOT = "/content/drive/MyDrive/thesis_data"
OUTPUT_FILE = "stability_perplexity_matrix_REPLAY.csv"

In [ ]:
def load_val_data(step_num):
    """
    Loads the validation set for a specific temporal step.
    Your structure: thesis_data/step_1/val.jsonl, step_2/val.jsonl, etc.
    """
    path = f"{DATA_ROOT}/step_{step_num}/val.jsonl"

    data = []
    try:
        with open(path, 'r', encoding='utf-8') as f:
            for line in f:
                item = json.loads(line.strip())
                # Check that the required fields exist
                if 'question' in item and 'answer' in item:
                    data.append(item)
        print(f"Loaded {len(data)} examples from step_{step_num}")
    except FileNotFoundError:
        print(f"Warning: Validation file not found at {path}")
    except Exception as e:
        print(f"Error loading {path}: {e}")

    return data

# PERPLEXITY CALCULATION
def calculate_ppl_causal(model, tokenizer, data, max_length=512):
    """
    Calculates Perplexity for Causal LLMs (Phi, Pythia, Neo).
    Only calculates loss on the ANSWER tokens, not the question.
    This is the correct way to evaluate generative models.
    """
    nlls = []
    device = model.device

    for item in tqdm(data, desc="Calc PPL (Causal)", leave=False):
        question = item['question']
        answer = item['answer']

        # Format as used during training
        full_text = f"Question: {question}\nAnswer: {answer}"

        # Tokenize the full sequence
        inputs = tokenizer(full_text, return_tensors="pt", truncation=True, max_length=max_length)
        input_ids = inputs.input_ids.to(device)

        # Also tokenize just the question part to find where answer starts
        question_part = f"Question: {question}\nAnswer:"
        question_inputs = tokenizer(question_part, return_tensors="pt", truncation=True, max_length=max_length)
        question_length = question_inputs.input_ids.shape[1]

        # Create labels: -100 for question tokens (ignored in loss), actual tokens for answer
        labels = input_ids.clone()
        labels[0, :question_length] = -100  # Mask out the question part

        with torch.no_grad():
            outputs = model(input_ids, labels=labels)
            neg_log_likelihood = outputs.loss

        if not torch.isnan(neg_log_likelihood) and not torch.isinf(neg_log_likelihood):
            nlls.append(neg_log_likelihood)

    if not nlls:
        return float('inf')

    # Average NLL across the dataset and convert to perplexity
    ppl = torch.exp(torch.stack(nlls).mean())
    return ppl.item()

def calculate_ppl_seq2seq(model, tokenizer, data, max_length=512):
    """
    Calculates Perplexity for Seq2Seq models (T5).
    Input: Question. Labels: Answer.
    """
    nlls = []
    device = model.device

    for item in tqdm(data, desc="Calc PPL (Seq2Seq)", leave=False):
        # T5 was trained with this format
        input_text = f"Answer this constitutional law question: {item['question']}"
        target_text = item['answer']

        inputs = tokenizer(
            input_text,
            return_tensors="pt",
            truncation=True,
            max_length=max_length, 
        ).input_ids.to(device)

        labels = tokenizer(
            target_text,
            return_tensors="pt",
            truncation=True,
            max_length=max_length
        ).input_ids.to(device)

        with torch.no_grad():
            outputs = model(input_ids=inputs, labels=labels)
            neg_log_likelihood = outputs.loss

        if not torch.isnan(neg_log_likelihood) and not torch.isinf(neg_log_likelihood):
            nlls.append(neg_log_likelihood)

    if not nlls:
        return float('inf')

    ppl = torch.exp(torch.stack(nlls).mean())
    return ppl.item()

In [ ]:
def main():
    results = []
    # Iterate through Model Families
    for family_name, steps_dict in model_families.items():
        print(f"Evaluating Family: {family_name}")
        is_seq2seq = "t5" in family_name.lower()

        # Iterate through Training Steps (Model Checkpoints)
        for train_step, model_id in sorted(steps_dict.items()):
            print(f"Loading Model trained through Step {train_step}: {model_id}")
            # Load Model & Tokenizer
            try:
                tokenizer = AutoTokenizer.from_pretrained(model_id, token=hf_token, trust_remote_code=True)

                if is_seq2seq:
                    model = AutoModelForSeq2SeqLM.from_pretrained(
                        model_id,
                        torch_dtype=torch.float16,
                        token=hf_token,
                        trust_remote_code=True
                    ).to("cuda")
                else:
                    model = AutoModelForCausalLM.from_pretrained(
                        model_id,
                        torch_dtype=torch.float16,
                        token=hf_token,
                        trust_remote_code=True
                    ).to("cuda")
                    if tokenizer.pad_token is None:
                        tokenizer.pad_token = tokenizer.eos_token

                model.eval()
                print("Model loaded successfully")

            except Exception as e:
                print(f"Failed to load {model_id}: {e}")
                continue

            # Iterate through ALL Data Steps (Test on all temporal periods)
            for data_step in range(1, 11):
                print(f"\n  Evaluating on Data Step {data_step}...")
                val_data = load_val_data(data_step)

                if not val_data:
                    print(f"    Skipping (no data found)")
                    results.append({
                        "Model_Family": family_name,
                        "Training_Step": train_step,
                        "Data_Step": data_step,
                        "Perplexity": float('inf'),
                        "Status": "No Data"
                    })
                    continue

                # Calculate PPL
                try:
                    if is_seq2seq:
                        ppl = calculate_ppl_seq2seq(model, tokenizer, val_data)
                    else:
                        ppl = calculate_ppl_causal(model, tokenizer, val_data)

                    print(f"Perplexity: {ppl:.4f}")
                    status = "Success"

                except Exception as e:
                    print(f"Error calculating PPL: {e}")
                    ppl = float('inf')
                    status = f"Error: {str(e)[:50]}"

                # Save Result
                results.append({
                    "Model_Family": family_name,
                    "Training_Step": train_step,
                    "Data_Step": data_step,
                    "Perplexity": ppl,
                    "Status": status
                })

            # Cleanup to save VRAM
            del model
            del tokenizer
            torch.cuda.empty_cache()

            # Save intermediate results after each model
            df = pd.DataFrame(results)
            df.to_csv(OUTPUT_FILE, index=False)

    # Final save and summary
    df = pd.DataFrame(results)
    df.to_csv(OUTPUT_FILE, index=False)
    print(f"Results saved to: {OUTPUT_FILE}")

    # Print summary statistics
    print("\nSummary by Model Family:")
    summary = df[df['Status'] == 'Success'].groupby('Model_Family').agg({
        'Perplexity': ['mean', 'min', 'max', 'std']
    }).round(2)
    print(summary)

if __name__ == "__main__":
    main()